# LoVoCCS morphology: Manually assigning broad ICM morphology classes

As part of the hot-phase intra-cluster medium (ICM) morphology subsection of the X-LoVoCCS project, we manually assign broad morphology classifications to our galaxy clusters - this helps us to place our quantitative measurements of morphology (centroid-shift, concentration, cluster component offsets and alignments) into perspective. This is a qualitative process as it relies on our experience with X-ray emissions from galaxy clusters, but we think they should be a reliable place to start for our morphological analyses. 

Very little code is executed in this notebook, as it mostly acts as a place for us to examine the images, make notes, and give them their classification. Here we list the broad morphology classes we can assign (top level bullet points), and optional modifiers that can be applied to those broad classifications (indented bullet points):

* **Centrally peaked (CeP)** - ICM that has bright (relative to the overall ICM) emission from a small region relatively close to the apparent centre of the ICM, and that appears approximately circular, without much additional structure.
    * ***Plateau (+Pl)*** - Centrally peaked emission that covers a larger fraction of the apparent size of the ICM than a typical **CP**-class.
* **Structured (S)** - X-ray emission that is not completely dominated by a bright central emission, but instead has other obvious structures such as spiral-like features, or hard drop-offs in surface brightness. Specific features can be indicated with optional modifiers (these can also be applied to the other broad morphological classes).
    * ***Sharp drop (+SD)*** - A sharp change in surface brightness, resulting in an obvious edge.
    * ***Spiral (+Sp)*** - ICM feature that resembles part of a spiral shape, commonly associated with sloshing.
    * ***Cavities (+Ca)*** - Obvious 'voids' of lower surface brightness, often close to the centre, and perhaps symmetrical - commonly associated with radio mode AGN in central galaxies.
    * ***Outer oslands (+OIs)*** - Areas of increased surface brightness, moderate size compared to the overall emission but set within it, and _not_ located in the centre of the extended emission.
* **Multi-modal (MM)** - ICM with multiple extended components of similar size that present as extended local peaks in a wider, mingled, extended emission.
    * ***Number of components (+NC\<NUMBER\>)*** - To indicate the number of major components of the multi-modal morphology.
* **Featureless blob (FB)** - The opposite end of the scale from a **CeP** classification, where the ICM X-ray surface brightness emission appears almost uniform across the majority of the ICM.

## Main takeaways 

In summary:

* 

## Import Statements

In [39]:
import pandas as pd
pd.set_option('display.max_columns', 500)

# This adds the directory above to the path, allowing me to import the common functions that I've written in
#  common.py - this just saves me repeating boring code and makes sure its all consistent
import sys
sys.path.insert(0, '../')
from common import lovoccs_cosmo

%matplotlib inline

## Useful values

Here we set up any values that are useful to several parts of the analysis in this notebook, and that we might wish to change in the future:

In [2]:
# These are the random states used for dimensionality reduction
umap_rand_state = 4059
tsne_rand_state = 907

## Output paths

Path to store the output manual classifications: 

In [3]:
# morph_fig_path = "../../outputs/figures/positions_and_morphology/morph_par_space/"
# os.makedirs(morph_fig_path, exist_ok=True)

## Cosmological model

We employ the same cosmological model utilized in the LoVoCCS-I & II analyses (Fu et al. [2022](https://ui.adsabs.harvard.edu/abs/2022ApJ...933...84F/abstract), [2024](https://ui.adsabs.harvard.edu/abs/2024ApJ...974...69F/abstract)):

In [4]:
lovoccs_cosmo

LambdaCDM(name=None, H0=<Quantity 71. km / (Mpc s)>, Om0=0.2648, Ode0=0.7352, Tcmb0=<Quantity 0. K>, Neff=3.04, m_nu=None, Ob0=0.0448)

## Loading data files

Important considerations for this dataset:

* Some clusters selected for LoVoCCS have been identified as multiple blended systems in the course of the X-LoVoCCS project - other LoVoCCS works may only have entries/have made measurements for the overall system.
* Not all LoVoCCS-II galaxy clusters have XMM data available - as such we will not yet have measured X-ray centroids for them.

### X-LoVoCCS-I base sample

In [5]:
xlovoccs_base = pd.read_csv("../../sample_files/X-LoVoCCSI.csv")
xlovoccs_base.insert(0, 'name', xlovoccs_base['LoVoCCSID'].apply(lambda x: "LoVoCCS-" + str(x)))
xlovoccs_base.head(6)

,name,LoVoCCSID,parent_LoVoCCSID,Name,start_ra,start_dec,MCXC_Redshift,MCXC_R500,MCXC_RA,MCXC_DEC,manual_xray_ra,manual_xray_dec,MCXC_Lx500_0.1_2.4
0,LoVoCCS-1,1,1,A2029,227.734300,5.745471,0.0766,1.3344,227.73000,5.720000,227.734300,5.745471,8.726709e+44
1,LoVoCCS-2,2,2,A401,44.740000,13.580000,0.0739,1.2421,44.74000,13.580000,NaN,NaN,6.088643e+44
2,LoVoCCS-4A,4A,4,A85North,10.458750,-9.301944,0.0555,1.2103,10.45875,-9.301944,NaN,NaN,5.100085e+44
3,LoVoCCS-4B,4B,4,A85South,10.451487,-9.460007,0.0555,1.2103,10.45875,-9.301944,10.451487,-9.460007,5.100085e+44
4,LoVoCCS-5,5,5,A3667,303.157313,-56.845978,0.0556,1.1990,303.13000,-56.830000,303.157313,-56.845978,4.871933e+44
5,LoVoCCS-7,7,7,A3827,330.480000,-59.950000,0.0980,1.1367,330.48000,-59.950000,NaN,NaN,4.204419e+44


## Defining an XGA ClusterSample

As we do not have observation cleaning turned on, we manually remove clusters that we know are excluded from our other analyses due to inadequate XMM data:

* LoVoCCS-41C is a component of 41 we identified from ROSAT Pointed data, but it does not fall on the XMM observation.
* LoVoCCS-33 is a cluster that partially falls on the edge of an observation of a nearby object - the coverage is insufficient for any real analysis however.

In [ ]:
xlovoccs_samp = xlovoccs_samp[~xlovoccs_samp['LoVoCCSID'].isin(['41C', '33'])]

We define a ClusterSample, centered on the 'start positions' we defined in the early stages of our analysis of this sample - the start positions will be the same as the MCXC positions in cases where we judged the coordinate to be adequate, but will be manually defined from modern observations if it was too far outside the main part of the ICM, or if the cluster has multiple components that went unresolved in MCXC:

In [ ]:
srcs = ClusterSample(xlovoccs_samp['start_ra'].values, xlovoccs_samp['start_dec'].values, xlovoccs_samp['MCXC_Redshift'].values, 
                     xlovoccs_samp['LoVoCCS_name'].values, r500=Quantity(xlovoccs_samp['MCXC_R500'].values, 'Mpc'), use_peak=False, 
                     clean_obs=False, cosmology=lovoccs_cosmo)
srcs.info()

## 